In [1]:
from flax import nnx
from jax import numpy as jnp

import torchvision
from torch.utils.data import random_split
import torch

import optax

from tqdm.notebook import trange

from models import LeNet
import jax_dataloader as jdl

/home/roman/repos/learning-ml/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
torch_rand = torch.Generator().manual_seed(42)


transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor(), lambda x: torch.einsum("chw->hwc", x)]) 
ds_train_full = torchvision.datasets.MNIST(root="/var/tmp", train=True, download=True, transform=transform)
ds_test = torchvision.datasets.MNIST(root="/var/tmp", train=False, transform=transform)
ds_train, ds_val = random_split(ds_train_full, [0.85, 0.15], torch_rand)

load = lambda x: jdl.DataLoader(x, 'pytorch', 64, True, generator=torch_rand)

dl_train, dl_val, dl_test = [load(x) for x in [ds_train, ds_val, ds_test]]

In [3]:
rngs = nnx.Rngs(42)
model = LeNet(rngs)
tx = optax.adam(1e-4)
optimiser = nnx.Optimizer(model, tx=tx, wrt=nnx.Param)
metrics = metrics = nnx.MultiMetric(
    accuracy=nnx.metrics.Accuracy(), loss=nnx.metrics.Average('loss')
)

In [4]:
nnx.tabulate(model, jnp.ones((1,28,28,1)))

                                               LeNet Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ path                ┃ type       ┃ inputs              ┃ outputs             ┃ Param                     ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│                     │ LeNet      │ float32[1,28,28,1]  │ float32[1,10]       │ 61,706 (246.8 KB)         │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ features            │ Sequential │ float32[1,28,28,1]  │ float32[1,5,5,16]   │ 2,572 (10.3 KB)           │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ features/layers/0   │ Conv       │ float32[1,28,28,1]  │ float32[1,28,28,6]  │ bias: float32[6]          │
│                     │            │                     │                     │ kernel: float32[5,5,1,6]  │
│                     │            │                     │                     │                           │
│                     │            │                     │                     │ 156 (624 B)               │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ features/layers/2   │ AvgPool    │ float32[1,28,28,6]  │ float32[1,14,14,6]  │                           │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ features/layers/3   │ Conv       │ float32[1,14,14,6]  │ float32[1,10,10,16] │ bias: float32[16]         │
│                     │            │                     │                     │ kernel: float32[5,5,6,16] │
│                     │            │                     │                     │                           │
│                     │            │                     │                     │ 2,416 (9.7 KB)            │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ features/layers/5   │ AvgPool    │ float32[1,10,10,16] │ float32[1,5,5,16]   │                           │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ classifier          │ Sequential │ float32[1,400]      │ float32[1,10]       │ 59,134 (236.5 KB)         │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ classifier/layers/0 │ Linear     │ float32[1,400]      │ float32[1,120]      │ bias: float32[120]        │
│                     │            │                     │                     │ kernel: float32[400,120]  │
│                     │            │                     │                     │                           │
│                     │            │                     │                     │ 48,120 (192.5 KB)         │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ classifier/layers/2 │ Linear     │ float32[1,120]      │ float32[1,84]       │ bias: float32[84]         │
│                     │            │                     │                     │ kernel: float32[120,84]   │
│                     │            │                     │                     │                           │
│                     │            │                     │                     │ 10,164 (40.7 KB)          │
├─────────────────────┼────────────┼─────────────────────┼─────────────────────┼───────────────────────────┤
│ classifier/layers/4 │ Linear     │ float32[1,84]       │ float32[1,10]       │ bias: float32[10]         │
│                     │            │                     │                     │ kernel: float32[84,10]    │
│                     │            │                     │                  

''

In [4]:
nnx.tabulate(model, jnp.ones((1,28,28,1)))

                                      LeNet Summary                                      
┏━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ path  ┃ type   ┃ inputs             ┃ outputs             ┃ Param                     ┃
┡━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       │ LeNet  │ float32[1,28,28,1] │ float32[1,10]       │ 61,706 (246.8 KB)         │
├───────┼────────┼────────────────────┼─────────────────────┼───────────────────────────┤
│ conv1 │ Conv   │ float32[1,28,28,1] │ float32[1,28,28,6]  │ bias: float32[6]          │
│       │        │                    │                     │ kernel: float32[5,5,1,6]  │
│       │        │                    │                     │                           │
│       │        │                    │                     │ 156 (624 B)               │
├───────┼────────┼────────────────────┼─────────────────────┼───────────────────────────┤
│ conv2 │ Conv   │ float32[1,14,14,6] │ float32[1,10,10,16] │ bias: float32[16]         │
│       │        │                    │                     │ kernel: float32[5,5,6,16] │
│       │        │                    │                     │                           │
│       │        │                    │                     │ 2,416 (9.7 KB)            │
├───────┼────────┼────────────────────┼─────────────────────┼───────────────────────────┤
│ fc1   │ Linear │ float32[1,400]     │ float32[1,120]      │ bias: float32[120]        │
│       │        │                    │                     │ kernel: float32[400,120]  │
│       │        │                    │                     │                           │
│       │        │                    │                     │ 48,120 (192.5 KB)         │
├───────┼────────┼────────────────────┼─────────────────────┼───────────────────────────┤
│ fc2   │ Linear │ float32[1,120]     │ float32[1,84]       │ bias: float32[84]         │
│       │        │                    │                     │ kernel: float32[120,84]   │
│       │        │                    │                     │                           │
│       │        │                    │                     │ 10,164 (40.7 KB)          │
├───────┼────────┼────────────────────┼─────────────────────┼───────────────────────────┤
│ fc3   │ Linear │ float32[1,84]      │ float32[1,10]       │ bias: float32[10]         │
│       │        │                    │                     │ kernel: float32[84,10]    │
│       │        │                    │                     │                           │
│       │        │                    │                     │ 850 (3.4 KB)              │
├───────┼────────┼────────────────────┼─────────────────────┼───────────────────────────┤
│       │        │                    │               Total │ 61,706 (246.8 KB)         │
└───────┴────────┴────────────────────┴─────────────────────┴───────────────────────────┘
                                                                                         
                           Total Parameters: 61,706 (246.8 KB)                           

''

In [ ]:
@nnx.jit
def forward(model, metrics, X: jnp.ndarray, y):
    y_hat = model(X)
    loss = optax.softmax_cross_entropy_with_integer_labels(y_hat, y).mean()
    metrics.update(loss=loss, logits=y_hat, labels=y)
    return loss

@nnx.jit
def train_step(model: nnx.Module, metrics: nnx.MultiMetric, optimiser: nnx.Optimizer, X, y):
    grads = nnx.grad(forward)(model, metrics, X, y)
    optimiser.update(model, grads)
    return

In [ ]:
steps = 100

with trange(steps) as t:
    for i in t:
        model.train()
        metrics.reset()
        for X, y in dl_train:
            train_step(model, metrics, optimiser, X, y)

        m_t = metrics.compute()

        model.eval()
        metrics.reset()
        for X, y in dl_val:
            forward(model, metrics, X, y)

        m_v = metrics.compute()

        t.set_description(f"accuracy: train={m_t["accuracy"]:.2f} val={m_v["accuracy"]:.2f}")


  0%|          | 0/100 [00:00<?, ?it/s]